[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zahra-Alavi/ESMfluc/blob/main/notebooks/ESMfluc.ipynb)

# ESMfluc: from sequence to signed influence

Paste a protein sequence or upload a single-sequence FASTA. This notebook predicts single-PB/multi-PB probabilities and shows the residues supplying evidence to those decisions. A PDB structure is optional and is used only for visualization.

**Before running:** select a GPU under **Runtime → Change runtime type**. ESM3 top-28 needs about **15.5 GB of model downloads** for three seeds; frozen ESM2 with attention needs about **2.9 GB**. Downloads are cached for this runtime.

This notebook never asks for a Zenodo token. It uses the model source archived with the weights, without requiring a GitHub login.

Three-seed mode retains bands supported by at least two seeds. Single-seed preview is faster, but does not establish seed stability. Bands describe evidence used by the model, not measured motion or causal interactions between residues.


In [ ]:
#@title 1. Choose a model and input
from pathlib import Path
import sys, os, re, json, hashlib, urllib.request, urllib.error, zipfile, subprocess, shutil, threading, time, html, csv

def run(command, progress=None, interval=30):
    """Run a command and show its output here; Colab hides output from subprocess.check_call."""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"})
    stop = threading.Event()
    def report():
        while not stop.wait(interval):
            print(progress(), flush=True)
    if progress:
        threading.Thread(target=report, daemon=True).start()
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        stop.set()
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)

MODEL = "esm3_top28_bilstm_attn" #@param ["esm3_top28_bilstm_attn", "esm3_top4_bilstm_attn", "esm3_frozen_bilstm_attn", "esm2_frozen_bilstm_attn", "esm2_top4_bilstm_attn", "esm2_top28_bilstm_attn", "esm2_frozen_linear"]
SINGLE_SEED_PREVIEW = False #@param {type:"boolean"}
UPLOAD_FASTA = False #@param {type:"boolean"}
PASTED_SEQUENCE = "SVGEGPDTKPTARGQPAAVGRVVVLSGPSAVGKSTVVRCLRERIPNLHFSVSATTRAPRPGEVDGVDYHFIDPTRFQQLIDQGELLEWAEIHGGLHRSGTLAQPVRAAAATGVPVLIEVDLAGARAIKKTMPEAVTVFLAPPSWQDLQARLIGRGTETADVIQRRLDTARIELAAQGDFDKVVVNRRLESACAELVSLLVGTAPGSP" #@param {type:"string"}
ALLOW_CPU_DEBUG = False #@param {type:"boolean"}

# Author-only option: a local copy of the deposit, including metadata.zip.
# Leave empty for normal Colab use. Do not put credentials here.
LOCAL_BUNDLE = ""
RECORD_ID = 23580685  # Zenodo record (DOI 10.5281/zenodo.23580685)
EXPECTED_MANIFEST = "67d33ee00696e9fa42a56a1cf16d79d097c527868af0da0e7b4d3d62710d0bae"
EXPECTED_METADATA = "c8f66837b131fb67b3a94002926f7be56edc10984dc6d4109daf91a554769f6f"
WORK = Path("/content/esmfluc") if Path("/content").exists() else Path.cwd() / "artifacts/colab/notebook"
WORK.mkdir(parents=True, exist_ok=True)
MODELS = Path(LOCAL_BUNDLE).resolve() if LOCAL_BUNDLE else WORK / "models"
MODELS.mkdir(parents=True, exist_ok=True)
SEEDS = [1] if SINGLE_SEED_PREVIEW else [1, 2, 3]
print("Model:", MODEL, "| seeds:", SEEDS)
print("Default sequence: guanylate kinase (207 residues). Replace it or upload your own.")


## 2. Fetch the verified release metadata

Downloads the list of model files and the study's model code, and checks that both match the published versions.


In [ ]:
#@title Fetch the release metadata
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_small(name, expected):
    target = MODELS / name
    if target.exists():
        if file_hash(target) != expected:
            raise RuntimeError(f"Existing file has an incorrect checksum: {target}")
        return target
    if LOCAL_BUNDLE:
        raise FileNotFoundError(f"The local bundle is missing {name}")
    url = f"https://zenodo.org/records/{RECORD_ID}/files/{name}?download=1"
    try:
        with urllib.request.urlopen(url, timeout=120) as response:
            data = response.read(32 * 1024 * 1024)
    except urllib.error.HTTPError as error:
        if error.code in (401, 403, 404):
            raise RuntimeError("The model record may still be an unpublished draft. Its authors must approve and publish it before public downloads work. Do not enter a personal token in this notebook.") from None
        raise
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError(f"Downloaded checksum mismatch: {name}")
    target.write_bytes(data)
    return target

fetch_small("manifest.json", EXPECTED_MANIFEST)
fetch_small("metadata.zip", EXPECTED_METADATA)
MANIFEST = json.loads((MODELS / "manifest.json").read_text())
with zipfile.ZipFile(MODELS / "metadata.zip") as archive:
    raw = archive.read("release/download_zenodo.py")
    expected = MANIFEST["source_code"]["file_sha256"]["release/download_zenodo.py"]
    if hashlib.sha256(raw).hexdigest() != expected:
        raise RuntimeError("Downloader source checksum mismatch")
    DOWNLOADER = WORK / "download_zenodo.py"
    DOWNLOADER.write_bytes(raw)
print("Verified release", MANIFEST["release_version"], "and archived model source.")


## 3. Set up Python 3.11

Installs the package versions used in the study. This takes a few minutes.


In [ ]:
#@title Install the isolated inference environment
REQUIREMENTS = '# Python 3.11; aligned with pyproject.toml.\nbiopython==1.83\nesm==3.2.1.post1\nh5py==3.12.1\nhttpx==0.28.1\nmatplotlib==3.9.1\nMDAnalysis==2.4.3\nnetworkx==3.3\nnumpy==1.26.4\npandas==2.2.2\npheatmap>=1,<2\nplotly==5.24.1\npsutil==5.9.8\nrequests==2.32.2\nremotezip>=0.12,<1\nsafetensors==0.4.3\nscikit-learn==1.5.0\nscipy==1.13.1\ntorch==2.3.0\ntransformers==4.41.1\n'
run([sys.executable, "-m", "pip", "install", "-q", "uv>=0.8,<1"])
import sysconfig
UV = str(Path(sysconfig.get_path("scripts")) / "uv")
ENV = WORK / "python311"
PYTHON = str(ENV / "bin/python")
if not Path(PYTHON).exists():
    run([UV, "venv", "--python", "3.11", "--seed", str(ENV)])
requirements = [line.strip() for line in REQUIREMENTS.splitlines() if line.strip() and not line.startswith("#")]
if not MODEL.startswith("esm3"):
    requirements = [line for line in requirements if not line.startswith("esm==")]
run([UV, "pip", "install", "--python", PYTHON, *requirements])
DEVICE = "cpu" if ALLOW_CPU_DEBUG else "cuda"
run([PYTHON, "-c",
    "import sys, torch; print('Python',sys.version.split()[0], '| PyTorch',torch.__version__); "
    + ("print('CPU debug mode: inference will be slower.')" if ALLOW_CPU_DEBUG else
       "assert torch.cuda.is_available(), 'Select a GPU runtime in Colab, then rerun this cell.'; print(torch.cuda.get_device_name(0)); "
       "major, minor = torch.cuda.get_device_capability(0); "
       "assert f'sm_{major}{minor}' in torch.cuda.get_arch_list(), "
       "f'This GPU ({torch.cuda.get_device_name(0)}) is not supported by the PyTorch version used in the study. '"
       "'Choose Runtime > Change runtime type > T4 GPU, then run all cells again.'")])


In [ ]:
#@title Prepare the inference adapter
# The model, contribution calculation and detector are loaded from the verified
# deposit. This adapter handles input, sequential seed inference and display.
RUNNER_SOURCE = '"""Sequence-only ESMfluc inference and display using the deposited model source.\n\nThis adapter does not retrain models or replace the study band detector.\nIt is embedded in ESMfluc.ipynb so the notebook does not need a GitHub token.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport hashlib\nimport html\nimport importlib.metadata\nimport io\nimport json\nfrom pathlib import Path\nimport sys\nfrom types import SimpleNamespace\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\n\nMANIFEST_SHA256 = "67d33ee00696e9fa42a56a1cf16d79d097c527868af0da0e7b4d3d62710d0bae"\nMETADATA_SHA256 = "c8f66837b131fb67b3a94002926f7be56edc10984dc6d4109daf91a554769f6f"\nSOURCES = (\n    "esmfluc/training/models.py", "esmfluc/inference/get_attn.py",\n    "esmfluc/analysis/bands/extract_signed_contribution_bands.py",\n    "esmfluc/analysis/bands/analyze_signed_band_seed_reproducibility.py",\n)\nRED, BLUE = "#c95c4b", "#347fa0"\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef parse_sequence(text):\n    lines = [line.strip() for line in text.splitlines() if line.strip()]\n    headers = [line for line in lines if line.startswith(">")]\n    if len(headers) > 1 or (headers and not lines[0].startswith(">")):\n        raise ValueError("Provide exactly one FASTA record or one plain sequence.")\n    name = headers[0][1:].strip()[:100] if headers else "query"\n    sequence = "".join("".join(line.split()) for line in lines if not line.startswith(">")).upper()\n    if not 3 <= len(sequence) <= 1024:\n        raise ValueError(f"The sequence has {len(sequence):,} residues; ESMfluc accepts 3–1,024. "\n                         "Sequences are never truncated or split, so run a region of at most 1,024 residues instead.")\n    invalid = set(sequence) - set("ACDEFGHIKLMNPQRSTVWYX")\n    if invalid:\n        raise ValueError(f"Unsupported sequence characters: {sorted(invalid)}. Use the 20 standard amino acids or X.")\n    return name or "query", sequence\n\n\ndef prepare_source(bundle, workspace):\n    """Verify the pinned deposit and extract only the necessary source files."""\n    bundle, workspace = Path(bundle), Path(workspace)\n    if sha256_file(bundle / "manifest.json") != MANIFEST_SHA256:\n        raise ValueError("The model manifest differs from this notebook\'s pinned release.")\n    if sha256_file(bundle / "metadata.zip") != METADATA_SHA256:\n        raise ValueError("Metadata archive checksum mismatch.")\n    manifest = json.loads((bundle / "manifest.json").read_text())\n    root = workspace / "source"\n    with zipfile.ZipFile(bundle / "metadata.zip") as archive:\n        names = list(SOURCES) + ["release/checkpoint_bundle.py"]\n        names += [n for n in archive.namelist() if n.startswith("upstream/")]\n        for name in names:\n            raw = archive.read(name)\n            expected = manifest["source_code"]["file_sha256"].get(name)\n            if name.startswith("upstream/"):\n                _, family, filename = name.split("/", 2)\n                expected = manifest["observed_upstream_cache"][family]["files"][filename]\n            if expected is None or hashlib.sha256(raw).hexdigest() != expected:\n                raise ValueError(f"Archived source/config checksum mismatch: {name}")\n            target = root / name\n            if not target.resolve().is_relative_to(root.resolve()):\n                raise ValueError("Unsafe archive filename")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(raw)\n    for parent in {parent for name in SOURCES for parent in (root / name).parents\n                   if parent != root and parent.is_relative_to(root)}:\n        (parent / "__init__.py").write_text(\'"""Archived ESMfluc inference package."""\\n\')\n    sys.path.insert(0, str(root))\n    sys.path.insert(0, str(root / "release"))\n    # A notebook run must not silently use a previously imported local module.\n    for name, module in list(sys.modules.items()):\n        if name.startswith("esmfluc.") and getattr(module, "__file__", None):\n            relative = str(Path(module.__file__).resolve()).replace("\\\\", "/")\n            for source in SOURCES:\n                if relative.endswith(source) and sha256_file(Path(module.__file__)) != manifest["source_code"]["file_sha256"][source]:\n                    raise ValueError(f"An incompatible module was already imported: {name}")\n    return manifest, root\n\n\ndef make_model(training_args, source):\n    import torch\n    from esmfluc.inference.get_attn import ESM3Wrapper, EsmSequenceTokenizer, ESM3_AVAILABLE\n    from esmfluc.training.models import BiLSTMWithSelfAttentionModel, ESMLinearTokenClassifier\n    if training_args["esm_model"].startswith("esm3"):\n        if not ESM3_AVAILABLE:\n            raise RuntimeError("Install esm==3.2.1.post1 in the isolated Python 3.11 environment.")\n        from esm.models.esm3 import ESM3\n        tokenizer = EsmSequenceTokenizer()\n        def unavailable_decoder(*args, **kwargs):\n            raise RuntimeError("This notebook supports sequence-only ESMfluc inference, not ESM3 generation.")\n        # Sequence-only forward uses only the sequence tokenizer. Constructing\n        # the full tokenizer collection would trigger an unnecessary HF download.\n        with torch.device("meta"):\n            backbone = ESM3Wrapper(ESM3(\n                d_model=1536, n_heads=24, v_heads=256, n_layers=48,\n                structure_encoder_fn=unavailable_decoder,\n                structure_decoder_fn=unavailable_decoder,\n                function_decoder_fn=unavailable_decoder,\n                tokenizers=SimpleNamespace(sequence=tokenizer)))\n    else:\n        from transformers import EsmConfig, EsmModel, EsmTokenizer\n        folder = source / "upstream/esm2"\n        config = EsmConfig.from_pretrained(folder, local_files_only=True)\n        tokenizer = EsmTokenizer.from_pretrained(folder, local_files_only=True)\n        with torch.device("meta"):\n            backbone = EsmModel(config)\n    with torch.device("meta"):\n        if training_args["architecture"] == "esm_linear":\n            model = ESMLinearTokenClassifier(backbone, num_classes=2)\n        else:\n            model = BiLSTMWithSelfAttentionModel(\n                backbone, training_args["hidden_size"], training_args["num_layers"],\n                num_classes=2, dropout=training_args["dropout"],\n                bidirectional=training_args["bidirectional"])\n    return model, tokenizer\n\n\ndef materialize_buffers(model):\n    """Initialize deterministic nonpersistent buffers absent from state_dict."""\n    import torch\n    for module in model.modules():\n        if type(module).__name__ == "RotaryEmbedding" and module.inv_freq.is_meta:\n            module.device = torch.device("cpu")\n            module.reset_parameters()\n    for name, buffer in list(model.named_buffers()):\n        if buffer.is_meta and name.endswith("embeddings.position_ids"):\n            parent, attribute = name.rsplit(".", 1)\n            model.get_submodule(parent)._buffers[attribute] = torch.arange(\n                buffer.shape[-1], dtype=buffer.dtype).reshape(buffer.shape)\n    if any(b.is_meta for _, b in model.named_buffers()):\n        raise RuntimeError("Unsupported nonpersistent model buffers; refusing an incomplete model.")\n\n\ndef detect_bands(influence, sequence, condition, seed):\n    from esmfluc.analysis.bands.extract_signed_contribution_bands import (\n        detect_raw_mad_amplitude_apices, build_apex_rows,\n    )\n    n = len(sequence)\n    calls = detect_raw_mad_amplitude_apices(influence, 0, n, 1, n - 1, 2.0)\n    context = dict(condition=condition, seed=seed, split="input", json_path="notebook",\n                   influence_field="signed_column_influence", protein="query",\n                   protein_length=n, eligible_start=1, eligible_end=n - 1, scale_tolerance=2)\n    missing = np.full(n, np.nan)\n    rows, _ = build_apex_rows(influence, missing, missing, sequence, calls, context,\n                              support_method="half_intensity_merge")\n    return rows\n\n\ndef stable_bands(mean_rows, seed_rows):\n    from esmfluc.analysis.bands.analyze_signed_band_seed_reproducibility import interval_iou_matches\n    if not mean_rows:\n        return []\n    mean_frame = pd.DataFrame(mean_rows)\n    support = {i: [] for i in mean_frame.index}\n    for seed, rows in seed_rows.items():\n        if not rows:\n            continue\n        frame = pd.DataFrame(rows)\n        for sign in (-1, 1):\n            matched = interval_iou_matches(mean_frame[mean_frame.sign == sign], frame[frame.sign == sign], .5)\n            for index in matched:\n                support[index].append(seed)\n    return [dict(row, seed_support_count=len(support[i]), supporting_seeds=support[i])\n            for i, row in enumerate(mean_rows) if len(support[i]) >= 2]\n\n\ndef public_bands(rows):\n    return [{"sign": int(r["sign"]),\n             "class_support": "multi-PB" if r["sign"] > 0 else "single-PB",\n             "peak_position": int(r["apex_residue_1based"]),\n             "peak_amino_acid": r["apex_amino_acid"],\n             "start": int(r["start_residue_1based"]),\n             "end": int(r["end_residue_1based_inclusive"]),\n             "peak_influence": float(r["apex_signed_column_influence"]),\n             "seed_support_count": r.get("seed_support_count"),\n             "supporting_seeds": r.get("supporting_seeds", [])} for r in rows]\n\n\ndef mean_predictions(probabilities):\n    mean = np.asarray(probabilities, dtype=np.float64).mean(axis=0)\n    return mean, (mean[:, 1] > .5).astype(int)\n\n\nLINKED_VIEWER_JS = r"""\nconst data = JSON.parse(document.getElementById(\'viewer-data\').textContent);\nconst plot = document.getElementById(\'contribution-plot\');\nconst thresholdInput = document.getElementById(\'threshold\');\nconst sourceInput = document.getElementById(\'source-position\');\nconst status = document.getElementById(\'selection-status\');\nconst sequenceButtons = Array.from(document.querySelectorAll(\'.seq-residue\'));\nconst bandShapes = (plot.layout.shapes || []).slice();\nlet selectedKey = null;\nlet viewer = null;\n\nfunction receiversForKey(matrix, key, threshold) {\n  if (!Number.isInteger(key) || key < 1 || key > data.sequence.length ||\n      !Number.isFinite(threshold) || threshold < 0) return [];\n  return matrix.map((row, i) => ({position: i + 1, value: Number(row[key - 1])}))\n    .filter(r => Number.isFinite(r.value) && r.value !== 0 && Math.abs(r.value) >= threshold)\n    .sort((a, b) => Math.abs(b.value) - Math.abs(a.value) || a.position - b.position);\n}\n\nfunction baseStructure() {\n  if (!viewer) return;\n  viewer.removeAllLabels();\n  viewer.setStyle({}, {cartoon: {color: \'#d4e6ce\'}});\n  for (const b of data.structure.bands) {\n    const color = b.sign > 0 ? \'#c95c4b\' : \'#347fa0\';\n    const peakColor = b.sign > 0 ? \'#650010\' : \'#00264d\';\n    viewer.setStyle({serial: b.atoms}, {cartoon: {color: color}});\n    if (b.peak !== null) {\n      viewer.addStyle({serial: [b.peak]}, {sphere: {radius: 0.5, color: peakColor}});\n      viewer.addLabel(b.label, {fontColor: peakColor, backgroundOpacity: 0, fontSize: 13},\n                      {serial: [b.peak]});\n    }\n  }\n}\n\nfunction selectSource(key) {\n  if (!data.has_contributions) return;\n  const threshold = thresholdInput.value.trim() === \'\' ? NaN : Number(thresholdInput.value);\n  if (!Number.isInteger(key) || key < 1 || key > data.sequence.length) {\n    status.textContent = \'Choose a source position within the input sequence.\';\n    return;\n  }\n  if (!Number.isFinite(threshold) || threshold < 0) {\n    status.textContent = \'Enter a finite, nonnegative contribution threshold.\';\n    return;\n  }\n  selectedKey = key;\n  sourceInput.value = key;\n  // The full, unclipped matrix is used, not the displayed color intensity.\n  const receivers = receiversForKey(plot.data[0].z, key, threshold);\n  const byPosition = new Map(receivers.map(r => [r.position, r.value]));\n  for (const el of sequenceButtons) {\n    const position = Number(el.dataset.pos);\n    el.classList.toggle(\'selected-source\', position === key);\n    el.classList.toggle(\'selected-receiver\', byPosition.has(position));\n    el.title = el.dataset.description + (byPosition.has(position)\n      ? \'; contribution from source \' + key + \': \' + byPosition.get(position).toPrecision(6) : \'\');\n  }\n  Plotly.relayout(plot, {shapes: bandShapes.concat([{\n    type: \'rect\', xref: \'x\', yref: \'y domain\', x0: key - .5, x1: key + .5,\n    y0: 0, y1: 1, fillcolor: \'rgba(0,127,134,0.12)\', line: {color: \'#007f86\', width: 2}\n  }])});\n  const body = document.getElementById(\'receiver-rows\');\n  body.replaceChildren();\n  for (const r of receivers) {\n    const tr = document.createElement(\'tr\');\n    const mapped = data.structure && data.structure.mapping[r.position - 1];\n    const pdbLabel = mapped && mapped.resolved\n      ? mapped.pdb_chain + \':\' + mapped.structure_amino_acid + mapped.pdb_residue + mapped.insertion_code\n      : data.structure ? \'unresolved\' : \'not supplied\';\n    for (const value of [data.sequence[r.position - 1] + r.position, r.value.toPrecision(6), pdbLabel]) {\n      const td = document.createElement(\'td\'); td.textContent = value; tr.appendChild(td);\n    }\n    body.appendChild(tr);\n  }\n  document.getElementById(\'receiver-count\').textContent = receivers.length;\n  let text = \'Source \' + data.sequence[key - 1] + key + \': \' + receivers.length +\n    \' receivers with |C(i,j)| ≥ \' + threshold.toPrecision(5) + \'.\';\n  if (viewer) {\n    baseStructure();\n    const atoms = data.structure.atom_serials;\n    const receiverAtoms = receivers.flatMap(r => atoms[String(r.position)] || []);\n    if (receiverAtoms.length) viewer.setStyle({serial: receiverAtoms}, {cartoon: {color: \'#ffcf4d\'}});\n    const sourceAtoms = atoms[String(key)] || [];\n    if (sourceAtoms.length) viewer.setStyle({serial: sourceAtoms}, {cartoon: {color: \'#007f86\'}});\n    const ca = data.structure.ca_serials[String(key)];\n    if (ca !== undefined) viewer.addStyle({serial: [ca]}, {sphere: {radius: .6, color: \'#007f86\'}});\n    viewer.render();\n  }\n  if (data.structure) {\n    const mappedCount = receivers.filter(r => data.structure.mapping[r.position - 1].resolved).length;\n    text += \' Structure: \' + mappedCount + \'/\' + receivers.length + \' receivers have mapped coordinates.\';\n    if (!data.structure.mapping[key - 1].resolved) text += \' The source has no resolved coordinates.\';\n  }\n  status.textContent = text;\n}\n\nif (data.structure && typeof $3Dmol !== \'undefined\') {\n  try {\n    viewer = $3Dmol.createViewer(\'structure-viewer\', {backgroundColor: \'white\'});\n    viewer.addModel(data.structure.pdb, \'pdb\'); baseStructure(); viewer.zoomTo(); viewer.render();\n  } catch (error) {\n    viewer = null;\n    document.getElementById(\'structure-note\').textContent =\n      \'The 3D viewer could not start: \' + String(error.message || error) +\n      \'. Sequence selection and numerical results remain available.\';\n  }\n} else if (data.structure) {\n  document.getElementById(\'structure-note\').textContent =\n    \'The 3D viewer requires access to 3Dmol.org. Sequence selection remains available.\';\n}\nif (data.has_contributions) {\n  for (const el of sequenceButtons) el.addEventListener(\'click\', () => selectSource(Number(el.dataset.pos)));\n  document.getElementById(\'select-source\').addEventListener(\'click\', () => selectSource(Number(sourceInput.value)));\n  sourceInput.addEventListener(\'keydown\', e => {if (e.key === \'Enter\') selectSource(Number(sourceInput.value));});\n  thresholdInput.addEventListener(\'input\', () => {if (selectedKey !== null) selectSource(selectedKey);});\n  plot.on(\'plotly_click\', event => {\n    if (!event.points || !event.points.length) return;\n    const point = event.points[0];\n    const trace = plot.data[point.curveNumber];\n    if (trace && trace.meta && trace.meta.role === \'source\') selectSource(Number(point.x));\n  });\n  document.getElementById(\'clear-selection\').addEventListener(\'click\', () => {\n    selectedKey = null;\n    for (const el of sequenceButtons) {\n      el.classList.remove(\'selected-source\', \'selected-receiver\'); el.title = el.dataset.description;\n    }\n    document.getElementById(\'receiver-rows\').replaceChildren();\n    document.getElementById(\'receiver-count\').textContent = \'0\';\n    Plotly.relayout(plot, {shapes: bandShapes}); baseStructure(); if (viewer) viewer.render();\n    status.textContent = \'Selection cleared. Click a heatmap column or a sequence residue to select its source.\';\n  });\n}\n"""\n\n\ndef write_linked_report(output, sequence, probabilities, matrix, influence, bands, stable, structure=None):\n    import plotly.graph_objects as go\n    from plotly.subplots import make_subplots\n    n = len(sequence)\n    positions = np.arange(1, n + 1)\n    label = "Seed-stable bands" if stable else "Single-seed candidate bands (stability not assessed)"\n    if matrix is None:\n        fig = go.Figure(go.Scatter(x=positions, y=probabilities[:, 1], mode="lines"))\n        fig.update_layout(template="plotly_white", title="Multi-PB probability — linear baseline",\n                          xaxis_title="Sequence position", yaxis_title="Probability")\n        limit = 0.0\n    else:\n        matrix = np.asarray(matrix)\n        if matrix.shape != (n, n) or not np.isfinite(matrix).all():\n            raise ValueError("Viewer contribution matrix must be finite and match the full input sequence")\n        limit = max(float(np.quantile(np.abs(matrix), .99)), 1e-12)\n        fig = make_subplots(rows=3, cols=1, row_heights=[.60, .25, .15], vertical_spacing=.08,\n                        subplot_titles=("Contribution matrix", f"Signed influence — {label}", "Predicted multi-PB probability"))\n        fig.add_trace(go.Heatmap(z=matrix, x=positions, y=positions, meta=dict(role="source"),\n            zmin=-limit, zmax=limit, colorscale="RdBu_r", colorbar=dict(title="Contribution", len=.5, y=.75),\n            hovertemplate="Key %{x} → query %{y}<br>Contribution %{z:.5g}<extra></extra>"), row=1, col=1)\n        fig.add_trace(go.Scatter(x=positions, y=influence, mode="lines", line=dict(color="#444"),\n            meta=dict(role="source"), hovertemplate="Key %{x}<br>Influence %{y:.5g}<extra></extra>", showlegend=False), row=2, col=1)\n        for band in bands:\n            color = RED if band["sign"] > 0 else BLUE\n            fig.add_vrect(x0=band["start"]-.5, x1=band["end"]+.5, fillcolor=color,\n                          opacity=.15, line_width=0, row=2, col=1)\n            p = band["peak_position"]\n            fig.add_trace(go.Scatter(x=[p], y=[influence[p-1]], mode="markers", meta=dict(role="source"),\n                marker=dict(color=color, size=7), showlegend=False,\n                text=[f"{band[\'peak_amino_acid\']}{p}; {band[\'class_support\']} support"],\n                hovertemplate="%{text}<br>Influence %{y:.5g}<extra></extra>"), row=2, col=1)\n        fig.add_trace(go.Scatter(x=positions, y=probabilities[:, 1], mode="lines",\n                                line=dict(color="#666"), showlegend=False), row=3, col=1)\n        fig.update_yaxes(autorange="reversed", title="Query residue i", row=1, col=1)\n        fig.update_xaxes(title="Key residue j", row=1, col=1)\n        fig.update_yaxes(zeroline=True, zerolinecolor="#888", title="Influence", row=2, col=1)\n        fig.update_yaxes(range=[0, 1], title="P(multi-PB)", row=3, col=1)\n        fig.update_xaxes(title="Sequence position", row=3, col=1)\n    fig.update_layout(template="plotly_white", height=1050 if matrix is not None else 400,\n                      margin=dict(l=65, r=65, t=55, b=45))\n    signs = np.zeros(n, dtype=int)\n    for band in bands:\n        signs[band["start"]-1:band["end"]] = band["sign"]\n    sequence_html = []\n    for p, aa in enumerate(sequence, 1):\n        cls = " positive-band" if signs[p-1] > 0 else " negative-band" if signs[p-1] < 0 else ""\n        description = f"{aa}{p}"\n        if structure and not structure["mapping"][p-1]["resolved"]:\n            cls += " unresolved"\n            description += "; no resolved structure coordinates"\n        sequence_html.append(f\'<button type="button" class="seq-residue{cls}" data-pos="{p}" \'\n            f\'data-description="{description}" title="{description}" \'\n            f\'{"disabled" if matrix is None else ""}><small>{p}</small>{aa}</button>\')\n    payload = json.dumps(dict(sequence=sequence, has_contributions=matrix is not None,\n                              structure=structure), allow_nan=False).replace("<", "\\\\u003c")\n    controls = f\'\'\'<section {\'hidden\' if matrix is None else \'\'}>\n<label>Source position j <input id="source-position" type="number" min="1" max="{n}" value="1"></label>\n<button id="select-source" type="button">Select source</button>\n<label>Minimum |contribution| <input id="threshold" type="number" min="0" step="any" value="{limit*.5:.12g}"></label>\n<button id="clear-selection" type="button">Clear selection</button>\n<p>Click a heatmap column or a sequence residue. Cyan marks the source; gold marks receivers meeting the threshold.\nSelection uses individual source columns, not whole bands. The threshold is a display control, not a statistical test.</p>\n<p id="selection-status" role="status" aria-live="polite">No source selected.</p>\n<details><summary>Receiving residues (<span id="receiver-count">0</span>)</summary>\n<div class="receiver-table"><table><thead><tr><th>Input residue</th><th>Signed contribution</th><th>Structure residue</th></tr></thead>\n<tbody id="receiver-rows"></tbody></table></div></details></section>\'\'\'\n    structure_html = \'\'\'<aside><div id="structure-viewer"></div><p id="structure-note">\nStructure labels use PDB numbering. Only mapped coordinates are highlighted; missing residues remain visible in the sequence.</p></aside>\'\'\' if structure else \'\'\n    structure_script = \'<script src="https://3Dmol.org/build/3Dmol-min.js"></script>\' if structure else \'\'\n    fragment = fig.to_html(full_html=False, include_plotlyjs=True, div_id="contribution-plot", post_script=LINKED_VIEWER_JS)\n    caption = ("Red contributions support multi-PB predictions; blue contributions support single-PB predictions. "\n               "Band colors in the sequence describe evidence supplied to queries, not the source residue\'s predicted class. "\n               "Colors saturate at the 99th percentile; selection uses full, unclipped contributions. "\n               "These are model contributions, not causal biological interactions.") if matrix is not None else (\n               "The linear baseline provides probabilities but has no task-specific contribution matrix or bands.")\n    page = \'\'\'<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>ESMfluc: sequence and residue contributions</title><style>\nbody{font:14px Arial,sans-serif;color:#222;margin:16px}button,input{font:inherit}section{margin-bottom:14px}\nlabel{display:inline-block;margin:4px 12px 4px 0}input{width:110px}button{cursor:pointer}\n.sequence{display:flex;flex-wrap:wrap;gap:2px;max-height:220px;overflow:auto;padding:5px;border:1px solid #ddd}\n.seq-residue{width:27px;border:1px solid transparent;background:#f4f4f4;padding:2px;color:#222}\n.seq-residue small{display:block;font:9px Arial,sans-serif}.positive-band{background:#f4d9d4}.negative-band{background:#d7e8f0}\n.unresolved{border-bottom:2px dotted #777}.selected-receiver{background:#ffcf4d;color:#111}\n.selected-source{background:#007f86!important;color:white!important;border:2px solid #003d41}\n.layout{display:grid;grid-template-columns:minmax(0,1.25fr) minmax(300px,1fr);gap:16px}\n.plots{grid-column:1;grid-row:1;min-width:0}aside{grid-column:2;grid-row:1}\n.sequence-only{grid-template-columns:minmax(0,1fr)}#structure-viewer{height:600px;position:relative;width:100%}\n.receiver-table{max-height:230px;overflow:auto}table{border-collapse:collapse}th,td{padding:4px 12px;border-bottom:1px solid #ddd;text-align:left}\n@media(max-width:850px){.layout{display:block}#structure-viewer{height:450px}}\n</style></head><body>\'\'\'+controls+\'<div class="sequence">\'+\'\'.join(sequence_html)+\'</div><p>\'+caption+\'</p>\'+structure_script+(\n        \'<script id="viewer-data" type="application/json">\'+payload+\'</script><div class="layout\'+(\'\' if structure else \' sequence-only\')+\'">\'+\n        structure_html+\'<div class="plots">\'+fragment+\'</div></div></body></html>\')\n    (Path(output) / "report.html").write_text(page)\n\n\ndef render_report(output, sequence, probabilities, matrix, influence, bands, stable):\n    write_linked_report(output, sequence, probabilities, matrix, influence, bands, stable)\n    if matrix is None:\n        return\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    n = len(sequence)\n    positions = np.arange(1, n + 1)\n    label = "Seed-stable bands" if stable else "Single-seed candidate bands (stability not assessed)"\n    limit = max(float(np.quantile(np.abs(matrix), .99)), 1e-12)\n    static, axes = plt.subplots(3, 1, figsize=(10, 10), gridspec_kw={"height_ratios": [5, 2, 1]}, constrained_layout=True)\n    image = axes[0].imshow(matrix, cmap="RdBu_r", vmin=-limit, vmax=limit, origin="upper",\n                            extent=(.5, n+.5, n+.5, .5), rasterized=True)\n    static.colorbar(image, ax=axes[0], label="Contribution")\n    axes[0].set(xlabel="Key residue j", ylabel="Query residue i")\n    axes[1].plot(positions, influence, color="#444", lw=1)\n    axes[1].axhline(0, color="#888", lw=.7)\n    for band in bands:\n        color = RED if band["sign"] > 0 else BLUE\n        axes[1].axvspan(band["start"]-.5, band["end"]+.5, color=color, alpha=.15)\n        p = band["peak_position"]\n        axes[1].scatter(p, influence[p-1], color=color, s=15, zorder=3)\n    axes[1].set(ylabel="Signed influence", title=label, xlim=(.5, n+.5))\n    axes[2].plot(positions, probabilities[:, 1], color="#666", lw=1)\n    axes[2].set(xlabel="Sequence position", ylabel="P(multi-PB)", ylim=(0, 1), xlim=(.5, n+.5))\n    for suffix in ("pdf", "svg"):\n        static.savefig(output / f"overview.{suffix}", facecolor="white")\n    plt.close(static)\n\n\ndef run(bundle, output, fasta, condition, seeds, device):\n    import torch\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    name, sequence = parse_sequence(Path(fasta).read_text())\n    if sorted(seeds) not in ([1], [1, 2, 3]):\n        raise ValueError("Choose seed 1 preview or all three seeds 1, 2, 3.")\n    if device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("No GPU is available. In Colab choose Runtime → Change runtime type → GPU.")\n    torch.set_num_threads(4)\n    manifest, source = prepare_source(Path(bundle), output / "runtime")\n    from checkpoint_bundle import load_bundle_state, model_entry\n    from esmfluc.inference.get_attn import run_model_bilstm_attn\n    entries = [model_entry(manifest, condition, seed) for seed in seeds]\n    required = {manifest["backbones"][entries[0]["backbone"]]["file"]:\n                manifest["backbones"][entries[0]["backbone"]]["sha256"]}\n    required.update({entry["file"]: entry["sha256"] for entry in entries})\n    for filename, digest in required.items():\n        print(f"Checking {filename}", flush=True)\n        if sha256_file(Path(bundle) / filename) != digest:\n            raise ValueError(f"Weight checksum mismatch: {filename}")\n    arrays, per_seed, predictions, margins, biases = {}, {}, [], [], []\n    matrices, attentions, profiles = [], [], []\n    for entry in entries:\n        seed = entry["seed"]\n        print(f"Running {condition}, seed {seed}", flush=True)\n        model, tokenizer = make_model(entry["training_args"], source)\n        # Files were verified above. Strict loading still checks tensor names/shapes.\n        state = load_bundle_state(Path(bundle), condition, seed, verify=False)\n        model.load_state_dict(state, strict=True, assign=True)\n        del state\n        materialize_buffers(model)\n        model.to(device).eval()\n        if hasattr(model, "attention"):\n            attention, tokens, _, _, probs, contribution = run_model_bilstm_attn(\n                model, tokenizer, sequence, device, return_logit_contributions=True)\n            if len(tokens) != len(sequence):\n                raise ValueError("Token/residue alignment failed")\n            probabilities = probs.numpy()\n            matrix = contribution["matrix"]\n            influence = matrix.astype(np.float64).mean(axis=0)\n            matrices.append(matrix.astype(np.float64))\n            attentions.append(attention.astype(np.float64))\n            profiles.append(influence)\n            margins.append(contribution["margin"])\n            biases.append(contribution["bias"])\n            per_seed[seed] = detect_bands(influence, sequence, condition, seed)\n            arrays.update({f"seed_{seed}_C": matrix, f"seed_{seed}_A": attention,\n                           f"seed_{seed}_s": contribution["intrinsic_signed_evidence"],\n                           f"seed_{seed}_I": influence})\n        else:\n            encoded = tokenizer(sequence, return_tensors="pt", add_special_tokens=False)\n            with torch.no_grad():\n                logits, _ = model(encoded["input_ids"].to(device), encoded["attention_mask"].to(device))\n            probabilities = logits.softmax(-1)[0].cpu().numpy()\n        if probabilities.shape != (len(sequence), 2) or not np.isfinite(probabilities).all():\n            raise ValueError("Invalid residue probability output")\n        predictions.append(probabilities)\n        arrays[f"seed_{seed}_probabilities"] = probabilities\n        del model, tokenizer\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    mean_prob, labels = mean_predictions(predictions)\n    matrix = np.mean(matrices, axis=0) if matrices else None\n    influence = matrix.mean(axis=0) if matrix is not None else None\n    rows = detect_bands(influence, sequence, condition, "mean") if matrix is not None else []\n    stable = len(seeds) == 3\n    if stable:\n        rows = stable_bands(rows, per_seed)\n    bands = public_bands(rows)\n    table = pd.DataFrame({"position": np.arange(1, len(sequence)+1), "amino_acid": list(sequence),\n                          "multi_pb_probability": mean_prob[:, 1], "predicted_class": labels,\n                          "class_name": np.where(labels == 1, "multi-PB", "single-PB")})\n    arrays["mean_probabilities"] = mean_prob\n    if matrix is not None:\n        if not np.allclose(matrix.sum(axis=1) + np.mean(biases), np.mean(margins, axis=0), atol=1e-5, rtol=1e-5):\n            raise ValueError("Ensemble margin reconstruction failed")\n        arrays.update(mean_C=matrix, mean_A=np.mean(attentions, axis=0), mean_I=influence)\n        table["mean_logit_margin"] = np.mean(margins, axis=0)\n        table["signed_influence"] = influence\n        table["influence_seed_sd"] = np.std(profiles, axis=0, ddof=1) if stable else np.nan\n    table.to_csv(output / "predictions.csv", index=False)\n    pd.DataFrame(bands, columns=["sign", "class_support", "peak_position", "peak_amino_acid",\n        "start", "end", "peak_influence", "seed_support_count", "supporting_seeds"]).to_csv(output / "bands.csv", index=False)\n    np.savez_compressed(output / "contributions.npz", **arrays)\n    summary = dict(name=name, sequence=sequence, condition=condition, seeds=seeds, device=device,\n        band_mode="seed-stable" if stable and matrix is not None else "candidate" if matrix is not None else "not available",\n        bands=bands, n_bands=len(bands), source_commit=manifest["source_code"]["commit"],\n        manifest_sha256=MANIFEST_SHA256, weights_sha256=required,\n        band_detection=manifest["band_detection"], classes=manifest["prediction"]["classes"],\n        environment={p: importlib.metadata.version(p) for p in\n                     ["torch", "numpy", "pandas", "scipy", "transformers", "safetensors", "plotly", "biopython"]\n                     + (["esm"] if condition.startswith("esm3") else [])},\n        notes=["Mean probabilities define ensemble class predictions.",\n               "Mean contributions are averaged after inference; mean(A*s) is not generally mean(A)*mean(s).",\n               "Bands describe model evidence, not causal residue interactions."])\n    (output / "run.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    render_report(output, sequence, mean_prob, matrix, influence, bands, stable)\n    archive_results(output)\n    print(f"Finished: {len(sequence)} residues, {len(bands)} bands. Results: {output}", flush=True)\n\n\ndef archive_results(output):\n    output = Path(output)\n    files = [p for p in output.iterdir() if p.is_file() and p.suffix != ".zip"]\n    with zipfile.ZipFile(output / "esmfluc_results.zip", "w", zipfile.ZIP_DEFLATED) as z:\n        for p in files:\n            z.write(p, p.name)\n\n\ndef map_structure(output, pdb_path, chain_id):\n    """Map to the first PDB model by sequence alignment, never residue offset."""\n    from Bio.PDB import PDBParser, PDBIO, Select\n    from Bio.Align import PairwiseAligner\n    from Bio.Data.PDBData import protein_letters_3to1_extended\n    output = Path(output)\n    summary = json.loads((output / "run.json").read_text())\n    sequence, bands = summary["sequence"], summary["bands"]\n    model = next(PDBParser(QUIET=True).get_structure("input", str(pdb_path)).get_models())\n    available = [c.id for c in model if any("CA" in r for r in c)]\n    if not chain_id and len(available) == 1:\n        chain_id = available[0]\n    if chain_id not in available:\n        raise ValueError(f"Choose a protein chain from {available}")\n    chain = model[chain_id]\n    residues = [r for r in chain if "CA" in r and r.get_resname().upper() in protein_letters_3to1_extended]\n    structure_seq = "".join(protein_letters_3to1_extended[r.get_resname().upper()] for r in residues)\n    aligner = PairwiseAligner(mode="global", match_score=2, mismatch_score=-1,\n                              open_gap_score=-8, extend_gap_score=-.5)\n    alignment = aligner.align(sequence, structure_seq)[0]\n    mapping = {}\n    for a, b in zip(alignment.aligned[0], alignment.aligned[1]):\n        mapping.update(zip(range(int(a[0]), int(a[1])), range(int(b[0]), int(b[1]))))\n    comparable = [(i, j) for i, j in mapping.items() if structure_seq[j] != "X"]\n    identity = sum(sequence[i] == structure_seq[j] for i, j in comparable) / max(1, len(comparable))\n    coverage = len(mapping) / len(sequence)\n    if identity < .9 or coverage < .8:\n        raise ValueError(f"Structure mapping rejected: identity={identity:.1%}, coverage={coverage:.1%}; require 90%/80%.")\n    class ChosenChain(Select):\n        def accept_model(self, m): return m.id == model.id\n        def accept_chain(self, c): return c.id == chain_id\n        def accept_residue(self, r): return r.id in {rr.id for rr in residues}\n    writer = PDBIO()\n    writer.set_structure(model)\n    buffer = io.StringIO()\n    writer.save(buffer, select=ChosenChain())\n    pdb_text = buffer.getvalue()\n    displayed = next(PDBParser(QUIET=True).get_structure("display", io.StringIO(pdb_text)).get_models())[chain_id]\n    rows, atom_serials, ca_serials = [], {}, {}\n    for i, amino_acid in enumerate(sequence):\n        if i not in mapping:\n            rows.append(dict(position=i+1, amino_acid=amino_acid, resolved=False, pdb_chain=None,\n                             pdb_residue=None, insertion_code=None, structure_amino_acid=None,\n                             sequence_match=None))\n            continue\n        residue = residues[mapping[i]]\n        normalized = displayed[residue.id]\n        atom_serials[i+1] = [int(a.serial_number) for a in normalized.get_unpacked_list()]\n        ca_serials[i+1] = int(normalized["CA"].serial_number)\n        rows.append(dict(position=i+1, amino_acid=amino_acid, resolved=True, pdb_chain=chain_id,\n            pdb_residue=int(residue.id[1]), insertion_code=residue.id[2].strip(),\n            structure_amino_acid=structure_seq[mapping[i]], sequence_match=amino_acid == structure_seq[mapping[i]]))\n    selections = []\n    for band in bands:\n        positions = range(band["start"], band["end"]+1)\n        peak = band["peak_position"]\n        mapped = rows[peak-1]\n        selections.append(dict(sign=band["sign"], atoms=[a for p in positions for a in atom_serials.get(p, [])],\n            peak=ca_serials.get(peak), label=(f"{mapped[\'structure_amino_acid\']}{mapped[\'pdb_residue\']}{mapped[\'insertion_code\']}"\n                                           if mapped["resolved"] else None)))\n    payload = json.dumps(dict(pdb=pdb_text, bands=selections)).replace("<", "\\\\u003c")\n    page = """<!doctype html><html><meta charset="utf-8"><body>\n<div id="protein" style="width:100%;height:620px;position:relative"></div>\n<script src="https://3Dmol.org/build/3Dmol-min.js"></script><script>\nconst data=PAYLOAD;const v=$3Dmol.createViewer(\'protein\',{backgroundColor:\'white\'});\nv.addModel(data.pdb,\'pdb\');v.setStyle({},{cartoon:{color:\'#d4e6ce\'}});\nfor(const b of data.bands){const c=b.sign>0?\'#c95c4b\':\'#347fa0\';const p=b.sign>0?\'#650010\':\'#00264d\';\nv.setStyle({serial:b.atoms},{cartoon:{color:c}});\nif(b.peak!==null){v.addStyle({serial:[b.peak]},{sphere:{radius:0.5,color:p}});\nv.addLabel(b.label,{fontColor:p,backgroundOpacity:0,fontSize:13},{serial:[b.peak]});}}\nv.zoomTo();v.render();</script>\n<p>Red: multi-PB-supporting bands; blue: single-PB-supporting bands. Dark spheres mark resolved peaks.\nLabels use PDB numbering. Unresolved residues are not invented. Rotate to inspect hidden peaks.</p></body></html>""".replace("PAYLOAD", payload)\n    (output / "structure.html").write_text(page)\n    pd.DataFrame(rows).to_csv(output / "structure_mapping.csv", index=False)\n    (output / "structure_mapping.json").write_text(json.dumps(dict(identity=identity, input_coverage=coverage,\n        chain=chain_id, structure_sha256=sha256_file(pdb_path), model="first", mapping=rows), indent=2))\n    with np.load(output / "contributions.npz", allow_pickle=False) as arrays:\n        write_linked_report(output, sequence, arrays["mean_probabilities"],\n            arrays["mean_C"] if "mean_C" in arrays else None,\n            arrays["mean_I"] if "mean_I" in arrays else None, bands,\n            summary.get("band_mode") == "seed-stable",\n            structure=dict(pdb=pdb_text, bands=selections, mapping=rows,\n                           atom_serials=atom_serials, ca_serials=ca_serials))\n    archive_results(output)\n    print(f"Mapped chain {chain_id}: {identity:.1%} identity, {coverage:.1%} resolved coverage", flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    sub = p.add_subparsers(dest="command", required=True)\n    inference = sub.add_parser("predict")\n    inference.add_argument("--bundle", required=True, type=Path)\n    inference.add_argument("--output", required=True, type=Path)\n    inference.add_argument("--fasta", required=True, type=Path)\n    inference.add_argument("--condition", default="esm3_top28_bilstm_attn")\n    inference.add_argument("--seeds", nargs="+", type=int, default=[1, 2, 3])\n    inference.add_argument("--device", choices=("cuda", "cpu"), default="cuda")\n    structure = sub.add_parser("map")\n    structure.add_argument("--output", required=True, type=Path)\n    structure.add_argument("--pdb", required=True, type=Path)\n    structure.add_argument("--chain", default="")\n    validate = sub.add_parser("validate")\n    validate.add_argument("--fasta", required=True, type=Path)\n    args = p.parse_args()\n    if args.command == "predict":\n        run(args.bundle, args.output, args.fasta, args.condition, args.seeds, args.device)\n    elif args.command == "map":\n        map_structure(args.output, args.pdb, args.chain)\n    else:\n        _, sequence = parse_sequence(args.fasta.read_text())\n        print(f"Valid input: {len(sequence)} residues")\n\n\nif __name__ == "__main__":\n    main()\n'
RUNNER = WORK / "esmfluc_runner.py"
RUNNER.write_text(RUNNER_SOURCE)
print("Adapter ready; model weights and archived scientific code are unchanged.")


## 4. Validate the sequence and download the selected weights

One sequence of 3–1,024 residues (the 20 standard amino acids or X). Longer sequences stop with an error.


In [ ]:
#@title Validate the sequence and download the weights
if UPLOAD_FASTA:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one single-sequence FASTA file.")
    input_bytes = next(iter(uploaded.values()))
else:
    input_bytes = PASTED_SEQUENCE.encode()
FASTA = WORK / "input.fasta"
FASTA.write_bytes(input_bytes)
run([PYTHON, str(RUNNER), "validate", "--fasta", str(FASTA)])

entries = [m for m in MANIFEST["models"] if m["condition"] == MODEL and m["seed"] in SEEDS]
if len(entries) != len(SEEDS):
    raise RuntimeError("The selected model/seeds are not in this release.")
base = MANIFEST["backbones"][entries[0]["backbone"]]
needed = base["size_bytes"] + sum(entry["size_bytes"] for entry in entries)
missing = [entry for entry in [base, *entries] if not (MODELS / entry["file"]).exists()]
missing_bytes = sum(entry["size_bytes"] for entry in missing)
if missing:
    print(f"Downloading model weights: {missing_bytes/1e9:.2f} GB")
else:
    print("Model weights already downloaded.")
if shutil.disk_usage(MODELS).free < missing_bytes + 2_000_000_000:
    raise RuntimeError("Not enough free disk space for these model files and outputs.")
def downloaded():
    done = 0
    for entry in missing:
        for path in (MODELS / entry["file"], MODELS / (entry["file"] + ".partial")):
            if path.exists():
                done += path.stat().st_size
                break
    return f"Downloaded {done/1e9:.2f} of {missing_bytes/1e9:.2f} GB"
if not LOCAL_BUNDLE:
    run([PYTHON, str(DOWNLOADER), "--record", str(RECORD_ID),
        "--condition", MODEL, "--seeds", *map(str, SEEDS), "--destination", str(MODELS)],
        progress=downloaded if missing else None)
print(f"Model weights ready ({needed/1e9:.2f} GB).")


## 5. Predict and trace the model's evidence

Runs each seed and averages their predictions and contributions. Attention models also show which residues contribute to each prediction, and the bands where that evidence concentrates. A red band supports multi-PB predictions elsewhere in the sequence, not necessarily at the band itself. In the report, click a source column or sequence residue, and adjust the contribution threshold.


In [ ]:
#@title Run the prediction
OUT = WORK / "outputs" / time.strftime("%Y%m%d-%H%M%S")
command = [PYTHON, str(RUNNER), "predict", "--bundle", str(MODELS),
    "--fasta", str(FASTA), "--output", str(OUT), "--condition", MODEL,
    "--seeds", *map(str, SEEDS), "--device", DEVICE]
run(command)
SUMMARY = json.loads((OUT / "run.json").read_text())
print(f"Finished: {len(SUMMARY['sequence'])} residues; {SUMMARY['n_bands']} bands; mode: {SUMMARY['band_mode']}.")
print("Outputs:", OUT)


In [ ]:
#@title Show the report
# Click a source column or sequence residue; adjust the contribution threshold in the report.
from IPython.display import display, HTML
def show_report(path, height):
    content = Path(path).read_text()
    display(HTML('<iframe srcdoc="' + html.escape(content, quote=True) +
        f'" style="width:100%;height:{height}px;border:0" allow="fullscreen"></iframe>'))

show_report(OUT / "report.html", 1150)
rows = list(csv.DictReader((OUT / "bands.csv").open()))
if rows:
    columns = ["class_support", "peak_amino_acid", "peak_position", "start", "end", "seed_support_count"]
    body = "<tr>" + "".join("<th>" + html.escape(c) + "</th>" for c in columns) + "</tr>"
    for row in rows:
        body += "<tr>" + "".join("<td>" + html.escape(str(row.get(c, ""))) + "</td>" for c in columns) + "</tr>"
    display(HTML("<h4>Band peaks and intervals (input-sequence numbering)</h4><table border='1' cellpadding='5'>" + body + "</table>"))
else:
    print("No bands passed the detector/stability rule, or the selected model is the linear baseline.")


## 6. Optional: add a structure to the linked viewer

After step 5 has finished, tick **ADD_STRUCTURE**, enter a **PDB_ID** (for example 1ZNX) and its **PDB_CHAIN**, then run the cell below (▶). To use your own file instead, leave PDB_ID empty: a **Choose Files** button appears under the cell. The structure must be for the same protein. The first coordinate model is used. Mapping requires at least 90% sequence identity and 80% resolved coverage; PDB numbering is not assumed to match sequence positions.

The updated report links the heatmap, sequence and structure. Click one source column or sequence residue: cyan marks the source, and gold marks receiving queries above the adjustable absolute-contribution threshold. Missing coordinates remain missing in 3D but the residues can still be selected in the sequence. Clear the selection to restore the red/blue bands and dark peak spheres. The structure does not change predictions.


In [ ]:
#@title Optional PDB structure view
ADD_STRUCTURE = False #@param {type:"boolean"}
PDB_ID = "" #@param {type:"string"}
PDB_CHAIN = "A" #@param {type:"string"}

if ADD_STRUCTURE:
    PDB = WORK / "structure.pdb"
    pdb_id = PDB_ID.strip().upper()
    if pdb_id:
        if not re.fullmatch(r"[0-9][A-Z0-9]{3}", pdb_id):
            raise ValueError("Enter a 4-character PDB ID, such as 1ZNX, or leave PDB_ID empty to upload a file.")
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=60) as response:
                PDB.write_bytes(response.read())
        except urllib.error.HTTPError as error:
            raise RuntimeError(f"Could not download {pdb_id} in PDB format (HTTP {error.code}). "
                "Very large structures are only distributed as mmCIF; upload a PDB file instead.") from None
        print(f"Downloaded {pdb_id} from the RCSB Protein Data Bank.")
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload one PDB file.")
        PDB.write_bytes(next(iter(uploaded.values())))
    run([PYTHON, str(RUNNER), "map", "--output", str(OUT),
        "--pdb", str(PDB), "--chain", PDB_CHAIN])
    mapping = json.loads((OUT / "structure_mapping.json").read_text())
    print(f"Alignment identity: {mapping['identity']:.1%}; resolved coverage: {mapping['input_coverage']:.1%}")
    show_report(OUT / "report.html", 1350)
else:
    print("Structure view skipped; sequence-only results are complete. To add one, tick ADD_STRUCTURE and run this cell again.")


## 7. Save the results

The ZIP contains probabilities, band coordinates, numerical matrices, an interactive HTML report, PDF/SVG plots for attention models and run provenance. If a structure was supplied, its mapped view and residue table are included too.

The interactive heatmap and receiver selection use the full input sequence. Longer inputs may take more time to render. The NPZ arrays retain the full numerical results. Sequence-only reports work without a structure; 3D rendering requires access to 3Dmol.org.

Outputs contain your input sequence and, if supplied, structural coordinates. Treat them accordingly if your input is confidential. No results are uploaded to Zenodo by this notebook.


In [ ]:
#@title Save the results
archive = OUT / "esmfluc_results.zip"
try:
    from google.colab import files
except ImportError:
    print("Download the results archive from:", archive)
else:
    files.download(str(archive))
